# التجربة التحفيزية: هل تكتشف الطرق التقليدية النقاط الضارة فعلاً؟

**قبل البدء:**
1. ضع هذا الدفتر والملف `overlap_experiment.py` في **المجلد نفسه**.
2. الشاحن موصول، ووضع **Performance/Turbo** مفعّل في Armoury Crate، والسكون معطّل.
3. شغّل الخلايا بالترتيب من الأعلى.

> الكود الحسابي موجود في `overlap_experiment.py`، والدفتر يستدعيه فقط. هذا ضروري حتى يعمل التوازي بشكل صحيح على Windows داخل Jupyter.

## ١. تثبيت المكتبات (مرة واحدة فقط)

In [ ]:
%pip install -q scikit-learn pandas numpy scipy matplotlib tabulate openml cleanlab joblib

## ٢. الاستيراد والتحقق من البيئة

In [ ]:
import os, sys, argparse, importlib
from pathlib import Path

assert Path("overlap_experiment.py").exists(), "ضع overlap_experiment.py في نفس مجلد الدفتر"
sys.path.insert(0, os.getcwd())
import overlap_experiment as oe
importlib.reload(oe)          # يلتقط أي تعديل على الملف دون إعادة تشغيل النواة

import pandas as pd
from IPython.display import Markdown, Image, display

print("Logical CPUs:", os.cpu_count())
print("Python:", sys.version.split()[0])

## ٣. دوال مساعدة
`make_args` تبني الإعدادات، و`show` تعرض الملخص والصور داخل الدفتر.

In [ ]:
def make_args(source="openml", max_datasets=72, seeds=5, n_jobs=10,
              fracs=(0.05, 0.10), max_rows=3000, max_features=200,
              out="results", summarize_only=False):
    return argparse.Namespace(source=source, max_datasets=max_datasets, seeds=seeds,
                              n_jobs=n_jobs, fracs=list(fracs), max_rows=max_rows,
                              max_features=max_features, out=out,
                              summarize_only=summarize_only)


def show(out="results", sections=("Overlap by pair type", "How often each option is the best")):
    out = Path(out)
    text = (out / "summary.md").read_text(encoding="utf-8")
    blocks = text.split("## ")
    display(Markdown(blocks[0]))
    for b in blocks[1:]:
        if any(b.startswith(s) for s in sections):
            display(Markdown("## " + b))
    for f in sorted(out.glob("fig_jaccard_k*.png")) + [out / "fig_delta.png"]:
        if f.exists():
            display(Image(filename=str(f), width=600))
    log = out / "failures.log"
    if log.exists() and log.stat().st_size:
        print(f"ملاحظات/أخطاء: {sum(1 for _ in open(log, encoding='utf-8'))} سطر في {log}")

## ٤. اختبار سريع دون إنترنت (أقل من دقيقة)
يتأكد أن كل شيء يعمل على جهازك.

In [ ]:
oe.run(make_args(source="sklearn", seeds=2, n_jobs=8, out="results_smoke"))
show("results_smoke")

## ٥. نسخة أولية على OpenML-CC18 (20 مجموعة × 3 بذور)
- التحميل الأول من OpenML يأخذ وقتاً، ثم يُحفظ في الذاكرة المؤقتة.
- **إن انقطع التشغيل** (أو أوقفته بزر ■)، أعد تشغيل الخلية نفسها فتُكمل من حيث توقفت.

In [ ]:
oe.run(make_args(source="openml", max_datasets=20, seeds=3, n_jobs=10, out="results"))
show("results")

## ٦. التجربة الكاملة (72 مجموعة × 5 بذور)
شغّلها إن كانت النتائج الأولية واعدة. تستخدم **نفس المجلد** فتعيد استخدام ما انتهى.

In [ ]:
oe.run(make_args(source="openml", max_datasets=72, seeds=5, n_jobs=10, out="results"))
show("results")

## ٧. عرض الجداول الكاملة

In [ ]:
display(Markdown(Path("results/summary.md").read_text(encoding="utf-8")))

In [ ]:
# جدول الاختبارات الإحصائية مرتباً حسب أكبر تحسن
tests = pd.read_csv("results/downstream_tests.csv")
tests.sort_values("mean_delta_vs_none", ascending=False).head(20)

## ٨. إعادة بناء الجداول والصور فقط (دون حساب)

In [ ]:
oe.run(make_args(out="results", summarize_only=True))
show("results")